# 🧠 Module 1 — From Machine Learning to Deep Learning
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Understand why deep learning outperforms traditional ML on complex tasks
- Implement a **perceptron** (the single neuron) from scratch using NumPy
- Understand **linear regression** as the foundation of neural networks
- Visualise what a decision boundary means
- Build intuition for **loss functions** and **gradient descent**

---

## 1.1 — Why Deep Learning?

Traditional machine learning relies on **hand-crafted features**: a human expert decides which characteristics of the data (e.g., edge frequency, colour histogram) should be fed into a model.

Deep learning learns features **automatically** from raw data, stacking layers of transformations:

```
Raw Pixels → Edges → Shapes → Objects → Prediction
  (input)   (layer1) (layer2)  (layer3)   (output)
```

This is why deep learning dominates image recognition, speech, text, and video.


## 1.2 — Linear Regression: The Simplest Neural Network

Linear regression is a single neuron with no activation function. Understanding it deeply gives you the intuition for everything that follows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

# ── Generate synthetic data ────────────────────────────────
# y = 2x + 1 + noise
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = 2 * X + 1 + np.random.randn(*X.shape) * 0.8

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.6, color='#4C72B0', label='Data points')
plt.plot(X, 2*X + 1, 'r--', linewidth=2, label='True line (y=2x+1)')
plt.xlabel("X"); plt.ylabel("y")
plt.title("Our Regression Dataset")
plt.legend(); plt.grid(alpha=0.3)
plt.show()


## 1.3 — Implementing Linear Regression from Scratch (NumPy)

Before using any framework, we build the key concepts manually. This is the most important step.

In [ ]:
# ── Manual implementation: the core training loop ─────────

class LinearRegressionNumPy:
    def __init__(self):
        self.w = np.random.randn()  # weight
        self.b = np.random.randn()  # bias
    
    def forward(self, X):
        """y_hat = w*X + b"""
        return self.w * X + self.b
    
    def loss(self, y_pred, y_true):
        """Mean Squared Error: average squared difference"""
        return np.mean((y_pred - y_true) ** 2)
    
    def backward(self, X, y_pred, y_true):
        """Compute gradients analytically"""
        n = len(X)
        error = y_pred - y_true
        dw = (2/n) * np.sum(error * X)   # ∂L/∂w
        db = (2/n) * np.sum(error)        # ∂L/∂b
        return dw, db
    
    def update(self, dw, db, lr=0.01):
        """Gradient descent step"""
        self.w -= lr * dw
        self.b -= lr * db

# ── Training loop ──────────────────────────────────────────
model = LinearRegressionNumPy()
losses = []

print(f"Before training: w={model.w:.3f}, b={model.b:.3f}")
print("-" * 45)

for epoch in range(200):
    y_pred = model.forward(X)
    loss   = model.loss(y_pred, y)
    dw, db = model.backward(X, y_pred, y)
    model.update(dw, db, lr=0.05)
    losses.append(loss)
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | w={model.w:.3f}, b={model.b:.3f}")

print("-" * 45)
print(f"After training:  w={model.w:.3f}, b={model.b:.3f}")
print(f"True values:     w=2.000, b=1.000")


In [ ]:
# ── Visualise: loss curve + learned line ──────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(losses, color='#DD8452', linewidth=2)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.set_title("Training Loss Over Time"); ax1.grid(alpha=0.3)

# Learned line vs data
y_final = model.forward(X)
ax2.scatter(X, y, alpha=0.5, color='#4C72B0', label='Data', zorder=2)
ax2.plot(X, y_final, 'r-', linewidth=2.5, label=f'Learned: y={model.w:.2f}x+{model.b:.2f}')
ax2.plot(X, 2*X+1, 'g--', linewidth=1.5, alpha=0.7, label='True: y=2x+1')
ax2.set_xlabel("X"); ax2.set_ylabel("y")
ax2.set_title("Learned vs True Line"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Linear Regression — Training Results", fontsize=13)
plt.tight_layout(); plt.show()


## 1.4 — The Perceptron: A Single Neuron for Classification

The **perceptron** is a linear classifier — the earliest neural network (Rosenblatt, 1958). It applies a step function to a linear combination of inputs.

```
Input X → [w·X + b] → step function → 0 or 1
```

Despite its simplicity, understanding the perceptron is essential — every deep network is made of millions of these units.


In [ ]:
# ── Perceptron from scratch ────────────────────────────────

class Perceptron:
    def __init__(self, n_features, lr=0.1):
        self.w  = np.zeros(n_features)
        self.b  = 0.0
        self.lr = lr
    
    def activation(self, z):
        return 1 if z >= 0 else 0
    
    def predict(self, X):
        return np.array([self.activation(np.dot(x, self.w) + self.b) for x in X])
    
    def train(self, X, y, epochs=20):
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_hat = self.activation(np.dot(xi, self.w) + self.b)
                error = yi - y_hat
                self.w += self.lr * error * xi
                self.b += self.lr * error
                errors += int(error != 0)
            accuracy = 1 - errors / len(y)
            history.append(accuracy)
        return history

# ── Linearly separable dataset ─────────────────────────────
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

X_clf, y_clf = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42)
X_clf = StandardScaler().fit_transform(X_clf)

# Train perceptron
p = Perceptron(n_features=2, lr=0.1)
history = p.train(X_clf, y_clf, epochs=30)

print(f"Final accuracy: {history[-1]*100:.1f}%")


In [ ]:
# ── Decision boundary visualisation ───────────────────────
def plot_decision_boundary(model, X, y, title):
    h = 0.02
    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    
    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    plt.scatter(X[:,0], X[:,1], c=y, cmap='RdBu', edgecolors='k', linewidth=0.5, s=50)
    plt.title(title, fontsize=13); plt.xlabel("Feature 1"); plt.ylabel("Feature 2")
    plt.grid(alpha=0.3); plt.show()

plot_decision_boundary(p, X_clf, y_clf, "Perceptron — Decision Boundary")


## 1.5 — Gradient Descent: Visualised

Gradient descent is the engine of all deep learning. Let's visualise it on a simple 2D loss surface.

In [ ]:
# ── Gradient descent visualisation ────────────────────────
def loss_surface(w):
    """A simple bowl-shaped loss function: L(w) = w^2 + 3"""
    return w**2 + 3

# Simulate gradient descent
w_vals = [5.0]   # start far from optimum
lr = 0.2
for _ in range(15):
    w = w_vals[-1]
    grad = 2 * w       # dL/dw = 2w
    w_vals.append(w - lr * grad)

w_range = np.linspace(-6, 6, 300)
plt.figure(figsize=(9, 4))

# Left: loss surface with descent path
plt.subplot(1, 2, 1)
plt.plot(w_range, loss_surface(w_range), 'b-', linewidth=2, label='Loss surface')
for i, w in enumerate(w_vals):
    c = plt.cm.Reds(i / len(w_vals))
    plt.scatter(w, loss_surface(w), color=c, s=80, zorder=5)
plt.plot([w_vals[0]], [loss_surface(w_vals[0])], 'r^', ms=10, label='Start')
plt.plot([w_vals[-1]], [loss_surface(w_vals[-1])], 'g*', ms=12, label='End')
plt.xlabel("Weight (w)"); plt.ylabel("Loss"); plt.title("Gradient Descent on Loss Surface")
plt.legend(); plt.grid(alpha=0.3)

# Right: w value converging over steps
plt.subplot(1, 2, 2)
plt.plot(w_vals, 'o-', color='#DD8452', linewidth=2, markersize=7)
plt.axhline(0, color='green', linestyle='--', label='Optimum (w=0)')
plt.xlabel("Step"); plt.ylabel("Weight value")
plt.title(f"Weight Converging (lr={lr})")
plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Starting w=5.0 → Final w≈{w_vals[-1]:.4f} (should be near 0)")


## 📝 Exercises

1. **Modify the perceptron**: Change the learning rate to `0.01` and `1.0`. How does convergence change?
2. **Loss functions**: Implement **Mean Absolute Error (MAE)** — `mean(|y_pred - y_true|)` — inside the `LinearRegressionNumPy` class. Which converges faster, MAE or MSE?
3. **Gradient descent**: Change `lr` in the visualisation to `0.9`. What happens? This is called **overshooting**. Try `lr=1.0` — what do you observe?
4. **XOR problem**: The perceptron cannot solve the XOR problem (non-linearly separable data). Create an XOR dataset and verify this for yourself.

---

## ✅ Module 1 Summary

| Concept | What it is |
|---|---|
| Linear regression | One neuron, no activation, predicts continuous values |
| Perceptron | One neuron, step activation, binary classifier |
| Loss function | Measures how wrong our prediction is |
| Gradient | Direction of steepest increase in loss |
| Gradient descent | Walk in the negative gradient direction to reduce loss |

**Next up → Module 2: Neural Networks & Backpropagation**
